In [1]:
# -*- coding: utf-8 -*-
"""using_llama_with_RAG.ipynb

Automatically generated by Colab.

Original file is located at
    https://colab.research.google.com/drive/1uhDRXOJ_IaDpOdSZEVTSpZrnnuQMK21X
"""

# !pip uninstall -y transformers accelerate peft bitsandbytes trl
# !pip install -q -U bitsandbytes accelerate peft transformers trl datasets
# !pip install faiss-cpu

"""
RAG-Enhanced Humor Generation System with Multi-Step Prompt Pipeline
Combines fine-tuned Llama 3 8B with retrieval-augmented generation

DEFAULT MODE: RAG + Multi-Step Pipeline (Best Quality)
"""

import os
import json
import torch
import numpy as np
from typing import List, Dict, Tuple
from dataclasses import dataclass
import pickle
from tqdm import tqdm
import random

from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel
from sentence_transformers import SentenceTransformer
import faiss

import wandb
from huggingface_hub import notebook_login

# Import multi-step prompt pipeline components
from prompt_pipeline import HumorPromptPipeline, PipelineConfig, LLMClient
from patterns import JOKE_PATTERNS

# ============================================================================
# HUGGINGFACE CHAT CLIENT ADAPTER
# ============================================================================

class HFChatClient(LLMClient):
    """Adapter to use RAGHumorGenerator with HumorPromptPipeline"""
    
    def __init__(self, generator: "RAGHumorGenerator"):
        self.g = generator
    
    def complete(self, prompt: str, *, temperature: float, max_tokens: int) -> str:
        """Complete a prompt using the generator's model"""
        return self.g._generate_joke_custom(
            prompt, 
            temperature=temperature, 
            max_new_tokens=max_tokens
        )


In [2]:
# ============================================================================
# CONFIGURATION
# ============================================================================

@dataclass
class RAGConfig:
    """Configuration for RAG system - Optimized for Multi-Step Pipeline"""

    # Model paths
    BASE_MODEL = "meta-llama/Meta-Llama-3-8B-Instruct"
    LORA_PATH = r"C:\Users\Anwender\humor-project\src\generation\llama3_humor-neu\llama3-humor-lora\checkpoint-2300"

    # RAG database
    JOKES_DATASET = r"C:\Users\Anwender\humor-project\data\humor_RAG_data_20000_RAG.jsonl"
    RAG_INDEX_PATH = "./rag_index"

    # Embedding model for retrieval
    EMBEDDING_MODEL = "sentence-transformers/all-MiniLM-L6-v2"

    # Retrieval settings (optimized)
    TOP_K = 3  # ✅ OPTIMIZED: Only retrieve what we use
    CANDIDATE_K = 12  # ✅ OPTIMIZED: 4x TOP_K for MMR
    LAMBDA = 0.4
    SIMILARITY_THRESHOLD = 0.2

    # Generation settings
    MAX_NEW_TOKENS = 256
    TEMPERATURE = 0.85
    TOP_P = 0.9
    USE_WANDB = False
    DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
    
    # Pipeline optimization settings
    PIPELINE_TEMP_STEP1 = 0.5  # ✅ Lower for faster associations
    PIPELINE_TEMP_STEP2 = 0.6  # ✅ Lower for faster imagery
    PIPELINE_MAX_TOKENS_STEP1 = 150
    PIPELINE_MAX_TOKENS_STEP2 = 200

config = RAGConfig()


In [3]:


# ============================================================================
# RAG DATABASE BUILDER
# ============================================================================

class HumorRAGDatabase:
    """
    Build and query a RAG database of joke patterns and examples
    """

    def __init__(self, config: RAGConfig):
        self.config = config
        self.embedding_model = None
        self.index = None
        self.jokes_data = []

        self.use_diversity = True
        self.lambda_param = config.LAMBDA

    def build_database(self, jokes_file: str):
        """Build the RAG database from training jokes"""

        print("🏗️  Building RAG Database...")

        # Load embedding model
        print(f"  Loading embedding model: {self.config.EMBEDDING_MODEL}")
        self.embedding_model = SentenceTransformer(self.config.EMBEDDING_MODEL)
        self.embedding_model.to(self.config.DEVICE)

        # Load jokes
        print(f"  Loading jokes from: {jokes_file}")
        self.jokes_data = self._load_jokes(jokes_file)
        print(f"  Loaded {len(self.jokes_data)} jokes")

        # Create embeddings
        print("  Creating embeddings...")
        embeddings = self._create_embeddings()

        # Build FAISS index
        print("  Building FAISS index...")
        self.index = self._build_faiss_index(embeddings)

        # Save database
        self._save_database()

        print(f"✓ RAG Database built with {len(self.jokes_data)} jokes\n")

    def _load_jokes(self, jokes_file: str) -> List[Dict]:
        """Load jokes from JSONL file"""
        jokes = []

        with open(jokes_file, 'r', encoding='utf-8') as f:
            for line in f:
                data = json.loads(line)
                joke_text = data['messages'][1]['content']  # Assistant's response
                word1 = data.get('word1', '')
                word2 = data.get('word2', '')

                jokes.append({
                    'joke': joke_text,
                    'word1': word1,
                    'word2': word2,
                    'constraint_text': f"{word1} {word2}",
                })

        return jokes

    def _create_embeddings(self) -> np.ndarray:
        """Create embeddings for all jokes"""

        # We'll embed both the joke and the constraint words for better retrieval
        texts_to_embed = []
        for joke_data in self.jokes_data:
            # Combine constraint words and joke for richer embeddings
            combined_text = f"{joke_data['constraint_text']}: {joke_data['joke']}"
            texts_to_embed.append(combined_text)

        # Create embeddings in batches
        embeddings = self.embedding_model.encode(
            texts_to_embed,
            batch_size=32,
            show_progress_bar=True,
            convert_to_numpy=True
        )

        return embeddings

    def _build_faiss_index(self, embeddings: np.ndarray):
        """Build FAISS index for fast similarity search"""

        dimension = embeddings.shape[1]

        # Use IndexFlatIP for inner product (cosine similarity after normalization)
        index = faiss.IndexFlatIP(dimension)

        # Normalize embeddings for cosine similarity
        faiss.normalize_L2(embeddings)

        # Add to index
        index.add(embeddings)

        return index

    def _save_database(self):
        """Save database to disk"""

        os.makedirs(self.config.RAG_INDEX_PATH, exist_ok=True)

        # Save FAISS index
        faiss.write_index(self.index, f"{self.config.RAG_INDEX_PATH}/faiss.index")

        # Save jokes data
        with open(f"{self.config.RAG_INDEX_PATH}/jokes_data.pkl", 'wb') as f:
            pickle.dump(self.jokes_data, f)

        print(f"  ✓ Database saved to {self.config.RAG_INDEX_PATH}")

    def load_database(self):
        """Load pre-built database from disk"""

        print("📂 Loading RAG Database...")

        # Load embedding model
        self.embedding_model = SentenceTransformer(self.config.EMBEDDING_MODEL)
        self.embedding_model.to(self.config.DEVICE)

        # Load FAISS index
        self.index = faiss.read_index(f"{self.config.RAG_INDEX_PATH}/faiss.index")

        # Load jokes data
        with open(f"{self.config.RAG_INDEX_PATH}/jokes_data.pkl", 'rb') as f:
            self.jokes_data = pickle.load(f)

        print(f"✓ Loaded database with {len(self.jokes_data)} jokes\n")

    # def retrieve(self, query: str, word1: str = None, word2: str = None, k: int = None) -> List[Dict]:
    #     """
    #     Retrieve similar jokes based on query

    #     Args:
    #         query: Search query (e.g., the word pair or a description)
    #         word1: First constraint word (optional, for better retrieval)
    #         word2: Second constraint word (optional)
    #         k: Number of results to return

    #     Returns:
    #         List of relevant jokes with similarity scores
    #     """

    #     if k is None:
    #         k = self.config.TOP_K

    #     # Construct query text
    #     if word1 and word2:
    #         query_text = f"{word1} {word2}: {query}"
    #     else:
    #         query_text = query

    #     # Create query embedding
    #     query_embedding = self.embedding_model.encode(
    #         [query_text],
    #         convert_to_numpy=True
    #     )

    #     # Normalize for cosine similarity
    #     faiss.normalize_L2(query_embedding)

    #     # Search
    #     scores, indices = self.index.search(query_embedding, k)

    #     # Gather results
    #     results = []
    #     for score, idx in zip(scores[0], indices[0]):
    #         if score >= self.config.SIMILARITY_THRESHOLD:
    #             joke_data = self.jokes_data[idx].copy()
    #             joke_data['similarity_score'] = float(score)
    #             results.append(joke_data)

    #     return results

    def retrieve(self, query: str, word1: str = None, word2: str = None, k: int = None):
        """Main retrieval method with optional diversity"""

        if k is None:
            k = self.config.TOP_K

        if self.use_diversity:
            # Use MMR for diversity
            return self.retrieve_with_diversity(
                query, word1, word2,
                k=k,
                candidate_k=k * 4,  # Get 4x candidates
                lambda_param=self.lambda_param
            )
        else:
            # Fall back to pure semantic search
            return self._semantic_search(query, word1, word2, k=k)

    def retrieve_with_diversity(
        self,
        query: str,
        word1: str = None,
        word2: str = None,
        k: int = 5,
        candidate_k: int = 20,
        lambda_param: float = 0.5
    ) -> List[Dict]:
        """
        Retrieve jokes using Maximal Marginal Relevance (MMR)

        Args:
            query: Search query
            word1, word2: Constraint words
            k: Number of diverse results to return (final output)
            candidate_k: Number of candidates to consider (larger pool)
            lambda_param: Balance between relevance (1.0) and diversity (0.0)
                        0.5 = equal weight to both

        Returns:
            List of k diverse, relevant jokes
        """

        # Step 1: Get candidates using semantic search
        candidates = self._semantic_search(query, word1, word2, k=candidate_k)

        if len(candidates) <= k:
            return candidates

        # Step 2: Diversify using MMR
        diverse_results = self._mmr_diversify(candidates, k, lambda_param)

        return diverse_results

    def _semantic_search(
        self,
        query: str,
        word1: str = None,
        word2: str = None,
        k: int = 20
    ) -> List[Dict]:
        """
        Pure semantic search - this is your current retrieve() method
        Gets top-k most similar jokes
        """

        # Construct query
        if word1 and word2:
            query_text = f"{word1} {word2}: {query}"
        else:
            query_text = query

        # Create query embedding
        query_embedding = self.embedding_model.encode(
            [query_text],
            convert_to_numpy=True
        )
        faiss.normalize_L2(query_embedding)

        # Search
        scores, indices = self.index.search(query_embedding, k)

        # Gather candidates
        candidates = []
        for score, idx in zip(scores[0], indices[0]):
            if score >= self.config.SIMILARITY_THRESHOLD:
                joke_data = self.jokes_data[idx].copy()
                joke_data['similarity_score'] = float(score)
                joke_data['idx'] = idx
                candidates.append(joke_data)

        return candidates

    def _mmr_diversify(
        self,
        candidates: List[Dict],
        k: int,
        lambda_param: float = 0.5
    ) -> List[Dict]:

        if len(candidates) <= k:
            return candidates

        # Get embeddings for all candidates
        candidate_texts = [c['joke'] for c in candidates]
        candidate_embeddings = self.embedding_model.encode(
            candidate_texts,
            convert_to_numpy=True,
            show_progress_bar=False
        )
        faiss.normalize_L2(candidate_embeddings)

        # Selected indices
        selected_indices = []
        selected_embeddings = []

        # Always start with the most relevant (highest similarity score)
        best_idx = 0  # First candidate is most similar to query
        selected_indices.append(best_idx)
        selected_embeddings.append(candidate_embeddings[best_idx])

        # Iteratively select k-1 more jokes
        for _ in range(k - 1):
            best_score = -np.inf
            best_candidate_idx = None

            # For each remaining candidate
            for idx, candidate in enumerate(candidates):
                if idx in selected_indices:
                    continue

                # Relevance: similarity to query (higher is better)
                relevance = candidate['similarity_score']

                # Diversity: dissimilarity to already selected (higher is better)
                # Calculate similarity to all selected jokes
                candidate_emb = candidate_embeddings[idx].reshape(1, -1)
                selected_emb_matrix = np.array(selected_embeddings)

                # Cosine similarity (dot product since normalized)
                similarities_to_selected = np.dot(selected_emb_matrix, candidate_emb.T).flatten()
                max_similarity_to_selected = np.max(similarities_to_selected)

                # MMR score
                mmr_score = (lambda_param * relevance -
                            (1 - lambda_param) * max_similarity_to_selected)

                if mmr_score > best_score:
                    best_score = mmr_score
                    best_candidate_idx = idx

            # Add best candidate
            if best_candidate_idx is not None:
                selected_indices.append(best_candidate_idx)
                selected_embeddings.append(candidate_embeddings[best_candidate_idx])

        # Return selected jokes in order of MMR selection
        return [candidates[idx] for idx in selected_indices]

In [4]:
# ============================================================================
# RAG-ENHANCED MODEL WITH MULTI-STEP PIPELINE
# ============================================================================

class RAGHumorGenerator:
    """Combines fine-tuned Llama with RAG and multi-step prompt pipeline"""

    def __init__(self, config: RAGConfig):
        self.config = config
        self.model = None
        self.tokenizer = None
        self.rag_db = None

    def load_model(self):
        """Load fine-tuned model with LoRA"""
        print("🤖 Loading Fine-tuned Model...")
        self.tokenizer = AutoTokenizer.from_pretrained(self.config.LORA_PATH, trust_remote_code=True)
        if self.tokenizer.pad_token is None:
            self.tokenizer.pad_token = self.tokenizer.eos_token
            self.tokenizer.pad_token_id = self.tokenizer.eos_token_id
        print("  ✓ Tokenizer loaded")

        from transformers import BitsAndBytesConfig
        bnb_config = BitsAndBytesConfig(
            load_in_4bit=True, bnb_4bit_quant_type="nf4",
            bnb_4bit_compute_dtype=torch.bfloat16, bnb_4bit_use_double_quant=True,
        )
        base_model = AutoModelForCausalLM.from_pretrained(
            self.config.BASE_MODEL, quantization_config=bnb_config,
            device_map="auto", trust_remote_code=True,
        )
        print("  ✓ Base model loaded")
        self.model = PeftModel.from_pretrained(base_model, self.config.LORA_PATH)
        self.model.eval()
        print("  ✓ LoRA weights loaded")
        print(f"  ✓ Model ready on {self.config.DEVICE}\n")

    def load_rag_database(self, build_if_missing: bool = True):
        """Load or build RAG database"""
        self.rag_db = HumorRAGDatabase(self.config)
        if os.path.exists(f"{self.config.RAG_INDEX_PATH}/faiss.index"):
            self.rag_db.load_database()
        elif build_if_missing:
            print("⚠️  RAG database not found, building new one...")
            self.rag_db.build_database(self.config.JOKES_DATASET)
        else:
            raise FileNotFoundError(f"RAG database not found at {self.config.RAG_INDEX_PATH}")
    
    def generate(self, word1: str, word2: str) -> str:
        """🎯 MAIN METHOD - Uses RAG + Multi-Step Pipeline (best quality)"""
        result = self.generate_with_rag(word1, word2, use_rag=True, use_pipeline=True)
        return result['best_joke']

    def generate_with_rag(
        self, word1: str, word2: str,
        use_rag: bool = True, use_pipeline: bool = True,
        num_candidates: int = 1, return_retrieved: bool = False
    ) -> Dict:
        """Generate joke with RAG and/or multi-step pipeline"""
        retrieved_jokes = []
        if use_rag and self.rag_db is not None:
            query = f"joke with {word1} and {word2}"
            retrieved_jokes = self.rag_db.retrieve(query=query, word1=word1, word2=word2, k=self.config.TOP_K)

        generated_jokes = []
        if use_pipeline:
            for _ in range(num_candidates):
                joke = self._generate_with_pipeline(word1, word2, retrieved_jokes)
                generated_jokes.append(joke)
        else:
            prompt = self._construct_prompt(word1, word2, retrieved_jokes)
            for _ in range(num_candidates):
                joke = self._generate_joke(prompt)
                generated_jokes.append(joke)

        result = {
            'word1': word1, 'word2': word2, 'generated_jokes': generated_jokes,
            'best_joke': generated_jokes[0] if generated_jokes else "",
            'used_rag': use_rag, 'used_pipeline': use_pipeline,
            'num_retrieved': len(retrieved_jokes)
        }
        if return_retrieved:
            result['retrieved_examples'] = retrieved_jokes
        return result

    def _generate_with_pipeline(self, word1: str, word2: str, retrieved_jokes: List[Dict]) -> str:
        """Generate using multi-step prompt pipeline"""
        retrieved_block = ""
        if retrieved_jokes:
            retrieved_block = "\n".join([f"- {j['joke']}" for j in retrieved_jokes[:3]])
        
        text = f"{word1} {word2}"
        pattern = random.choice(JOKE_PATTERNS)
        llm = HFChatClient(self)
        pipe = HumorPromptPipeline(llm, PipelineConfig(
            temperature_step1=self.config.PIPELINE_TEMP_STEP1,
            temperature_step2=self.config.PIPELINE_TEMP_STEP2,
            temperature_step3=self.config.TEMPERATURE,
            max_tokens_step1=self.config.PIPELINE_MAX_TOKENS_STEP1,
            max_tokens_step2=self.config.PIPELINE_MAX_TOKENS_STEP2,
            max_tokens_step3=self.config.MAX_NEW_TOKENS,
        ))
        return pipe.run(pattern=pattern, text=text, retrieved_examples=retrieved_block if retrieved_block else None)

    def _generate_joke_custom(self, prompt: str, temperature: float, max_new_tokens: int) -> str:
        """Custom generation for pipeline (per-call temp/tokens)"""
        messages = [{"role": "user", "content": prompt}]
        formatted_prompt = self.tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        inputs = self.tokenizer(formatted_prompt, return_tensors="pt").to(self.model.device)
        with torch.no_grad():
            outputs = self.model.generate(
                **inputs, max_new_tokens=max_new_tokens, temperature=temperature,
                top_p=self.config.TOP_P, do_sample=True,
                pad_token_id=self.tokenizer.pad_token_id, eos_token_id=self.tokenizer.eos_token_id,
            )
        generated = self.tokenizer.decode(outputs[0], skip_special_tokens=True)
        if "assistant" in generated:
            return generated.split("assistant")[-1].strip()
        return generated[len(formatted_prompt):].strip()

    def _construct_prompt(self, word1: str, word2: str, retrieved_jokes: List[Dict]) -> str:
        """Construct prompt with optional RAG context (single-step)"""
        base_instruction = f"Generate a funny joke that naturally includes both of these words: '{word1}' and '{word2}'. The joke should be creative, humorous, and incorporate both words seamlessly."
        if not retrieved_jokes:
            return base_instruction

        context = """You are a comedian. You are generating humor.

A joke is funny because multiple cognitive mechanisms work together at once.
Do NOT copy wording, phrasing, or structure from the examples.
Instead, analyze WHY they work.

For each example below, humor may come from one or more of the following factors:
- Expectation violation (what the listener predicts vs what actually happens)
- Internal contradiction or incongruity
- Reversal of roles, status, or intent
- Literal interpretation of figurative language
- Treating something serious as trivial, or trivial as serious
- Absurd logic presented as reasonable
- Social norm violation that is harmless
- Overly confident reasoning applied to something wrong
- Sudden re-interpretation of earlier information
- Misdirection followed by a clean punchline
- Emotional contrast (calm setup → extreme or petty conclusion)
- Timing and brevity (a sharp, minimal punchline)

Below are jokes that successfully combine several of these mechanisms.
For each one, mentally identify:
- What expectation is created
- What assumption is broken
- What logical or social rule is violated
- Why the violation feels safe or playful rather than confusing

Examples:"""
        for i, joke_data in enumerate(retrieved_jokes[:3], 1):
            context += f"Example {i}: {joke_data['joke']}\n"
        context += f"""\n- Create a NEW joke that naturally includes BOTH of these words: "{word1}" and "{word2}"
- The joke must use at least TWO humor mechanisms from the list above
- The punchline should reframe the setup in an unexpected but internally consistent way
- The humor should come from meaning, not wordplay alone
- Do NOT explain the joke
- Do NOT reference the examples
- DO NOT explain why is the joke funny.
- DO NOT elaborate on joke.

Generate only the joke text and nothing else., {base_instruction.lower()}"""
        return context

    def _generate_joke(self, prompt: str) -> str:
        """Generate single joke (traditional)"""
        messages = [{"role": "user", "content": prompt}]
        formatted_prompt = self.tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        inputs = self.tokenizer(formatted_prompt, return_tensors="pt").to(self.model.device)
        with torch.no_grad():
            outputs = self.model.generate(
                **inputs, max_new_tokens=self.config.MAX_NEW_TOKENS,
                temperature=self.config.TEMPERATURE, top_p=self.config.TOP_P, do_sample=True,
                pad_token_id=self.tokenizer.pad_token_id, eos_token_id=self.tokenizer.eos_token_id,
            )
        generated = self.tokenizer.decode(outputs[0], skip_special_tokens=True)
        if "assistant" in generated:
            return generated.split("assistant")[-1].strip()
        return generated[len(formatted_prompt):].strip()

    def batch_generate(self, word_pairs: List[Tuple[str, str]], use_rag: bool = True, use_pipeline: bool = True) -> List[Dict]:
        """Generate jokes for multiple word pairs"""
        results = []
        mode_str = "Pipeline" if use_pipeline else "Single-step"
        rag_str = "with RAG" if use_rag else "without RAG"
        print(f"🎭 Generating jokes ({mode_str}, {rag_str}) for {len(word_pairs)} word pairs...")
        for word1, word2 in tqdm(word_pairs):
            result = self.generate_with_rag(word1, word2, use_rag=use_rag, use_pipeline=use_pipeline)
            results.append(result)
        return results


In [5]:
# ============================================================================
# EVALUATION & COMPARISON
# ============================================================================

class RAGEvaluator:
    """Compare RAG vs non-RAG generation"""

    @staticmethod
    def compare_generations(generator: RAGHumorGenerator, test_pairs: List[Tuple[str, str]]):
        """Compare RAG-enhanced vs vanilla fine-tuned model"""

        print("🔬 Comparing RAG vs Non-RAG Generation\n")
        print("="*80)

        for word1, word2 in test_pairs:
            print(f"\n📝 Words: {word1}, {word2}")
            print("-"*80)

            # Without RAG
            result_no_rag = generator.generate_with_rag(
                word1, word2,
                use_rag=False,
                return_retrieved=False
            )

            print(f"\n[Without RAG]")
            print(f"{result_no_rag['best_joke']}")

            # With RAG
            result_with_rag = generator.generate_with_rag(
                word1, word2,
                use_rag=True,
                return_retrieved=True
            )

            print(f"\n[With RAG] (Retrieved {result_with_rag['num_retrieved']} examples)")
            print(f"{result_with_rag['best_joke']}")

            if result_with_rag.get('retrieved_examples'):
                print(f"\n[Retrieved Examples:]")
                for i, ex in enumerate(result_with_rag['retrieved_examples'][:2], 1):
                    print(f"  {i}. {ex['joke'][:100]}... (score: {ex['similarity_score']:.3f})")

            print("="*80)



In [6]:
# ============================================================================
# QUICK START - Run these cells in order:
# 1. Imports & Config (cells above)
# 2. This cell (initialize & test)
# ============================================================================

# Initialize generator
generator = RAGHumorGenerator(config)
generator.load_model()
generator.load_rag_database()

print("✅ Generator ready!\n")
print("=" * 80)

# ✅ SIMPLE: Generate with best settings (RAG + Multi-Step Pipeline)
print("1️⃣  SIMPLE USAGE (Best Quality - DEFAULT)")
print("=" * 80)
joke = generator.generate("coffee", "meeting")
print(f"Joke: {joke}\n")

# ✅ DETAILED: Get full result with metadata
print("=" * 80)
print("2️⃣  DETAILED USAGE (With Metadata)")
print("=" * 80)
result = generator.generate_with_rag("python", "debugging", use_rag=True, use_pipeline=True, return_retrieved=True)
print(f"Joke: {result['best_joke']}")
print(f"Retrieved: {result['num_retrieved']} examples")
print(f"Used Pipeline: {result['used_pipeline']}\n")

# 🔬 TESTING ONLY: Compare different modes
print("3️⃣  TESTING: Compare Modes - FULL OUTPUT")
print("=" * 80)

test_pairs = [("algorithm", "coffee")]

for word1, word2 in test_pairs:
    print(f"\n📝 Testing: {word1} + {word2}")
    print("=" * 80)
    
    # Test 1: No RAG, No Pipeline
    print("\n[1] Baseline (No RAG, No Pipeline)")
    print("-" * 40)
    r1 = generator.generate_with_rag(word1, word2, use_rag=False, use_pipeline=False)
    print(r1['best_joke'])
    
    # Test 2: RAG, No Pipeline  
    print("\n[2] RAG Only (No Pipeline)")
    print("-" * 40)
    r2 = generator.generate_with_rag(word1, word2, use_rag=True, use_pipeline=False)
    print(r2['best_joke'])
    
    # Test 3: RAG + Pipeline (DEFAULT/BEST)
    print("\n[3] ✅ BEST (RAG + Multi-Step Pipeline)")
    print("-" * 40)
    r3 = generator.generate_with_rag(word1, word2, use_rag=True, use_pipeline=True, return_retrieved=True)
    print(r3['best_joke'])
    print(f"\nℹ️  Retrieved {r3['num_retrieved']} examples, Used Pipeline: {r3['used_pipeline']}")

print("\n" + "=" * 80)



🤖 Loading Fine-tuned Model...
  ✓ Tokenizer loaded


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

  ✓ Base model loaded
  ✓ LoRA weights loaded
  ✓ Model ready on cuda

⚠️  RAG database not found, building new one...
🏗️  Building RAG Database...
  Loading embedding model: sentence-transformers/all-MiniLM-L6-v2


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


  Loading jokes from: C:\Users\Anwender\humor-project\data\humor_RAG_data_20000_RAG.jsonl
  Loaded 20000 jokes
  Creating embeddings...


Batches:   0%|          | 0/625 [00:00<?, ?it/s]

  Building FAISS index...
  ✓ Database saved to ./rag_index
✓ RAG Database built with 20000 jokes

✅ Generator ready!

1️⃣  SIMPLE USAGE (Best Quality - DEFAULT)
Joke: I told my boss he has the wrong email address. I told my boss he has the wrong email address. So he fired me. And I told my boss he has the wrong email address. And my boss said, “What do you mean?”  “I mean we’re having a meeting,” I replied.  “Well what’s the subject line?” asked my boss.  “I didn’t know I needed one,” I said.

2️⃣  DETAILED USAGE (With Metadata)
Joke: What do you call a python with a great bedside manner? A snake charmer!
Retrieved: 3 examples
Used Pipeline: True

3️⃣  TESTING: Compare Modes - FULL OUTPUT

📝 Testing: algorithm + coffee

[1] Baseline (No RAG, No Pipeline)
----------------------------------------
What does every programmer need? An algorithm for coffee

[2] RAG Only (No Pipeline)
----------------------------------------
There is no algorithm for puns. It’s all just a matter of brew-tifu